In [11]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 24
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


In [1]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import wbgapi as wb
wb.db = 2

# Check WB tariff indicators
tariff_candidates = [
    'TM.TAX.MRCH.SM.AR.ZS',  # Tariff rate, applied, simple mean
    'TM.TAX.MRCH.WM.AR.ZS',  # Tariff rate, applied, weighted mean
    'TM.TAX.MANF.SM.AR.ZS',  # Tariff rate on manufactures
]

for code in tariff_candidates:
    try:
        df = wb.data.DataFrame(code, time=range(2020, 2023), labels=False)
        print(f"IN WDI: {code} — shape {df.shape}")
    except:
        print(f"NOT IN WDI: {code}")

IN WDI: TM.TAX.MRCH.SM.AR.ZS — shape (266, 3)
IN WDI: TM.TAX.MRCH.WM.AR.ZS — shape (266, 3)
IN WDI: TM.TAX.MANF.SM.AR.ZS — shape (266, 3)


In [2]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import wbgapi as wb
wb.db = 2

check_indicators = {
    'Carbon pricing ETS': 'EN.CLC.GHGR.MT.CE',
    'Carbon tax': 'EN.CLC.CARP.ZS',
    'WTO TFA': 'TT.TRF.FACT.XD.ZS',
}

for name, code in check_indicators.items():
    try:
        df = wb.data.DataFrame(code, time=range(2020, 2023), labels=False)
        print(f"IN WDI: {name} ({code}) — shape {df.shape}")
    except:
        print(f"NOT IN WDI: {name} ({code})")

NOT IN WDI: Carbon pricing ETS (EN.CLC.GHGR.MT.CE)
NOT IN WDI: Carbon tax (EN.CLC.CARP.ZS)
NOT IN WDI: WTO TFA (TT.TRF.FACT.XD.ZS)


In [5]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
from datetime import datetime, timedelta

IMAPP_BASE = "https://www.elibrary-areaer.imf.org/Macroprudential/Documents"

def get_latest_imapp_url():
    """Try recent dates to find the latest iMaPP ZIP file."""
    # Try dates from today backwards for up to 2 years
    check_date = datetime.today()
    for _ in range(730):
        date_str = check_date.strftime("%Y-%m-%d")
        url = f"{IMAPP_BASE}/iMaPP_database-{date_str}.zip"
        try:
            r = requests.head(url, timeout=5, allow_redirects=True)
            if r.status_code == 200 and 'zip' in r.headers.get('Content-Type', '').lower():
                print(f"Found: {url}")
                return url, date_str
        except:
            pass
        check_date -= timedelta(days=1)
    return None, None

print("Searching for latest iMaPP ZIP...")
IMAPP_URL, IMAPP_DATE = get_latest_imapp_url()
print(f"Latest: {IMAPP_DATE}")

Searching for latest iMaPP ZIP...
Found: https://www.elibrary-areaer.imf.org/Macroprudential/Documents/iMaPP_database-2025-09-29.zip
Latest: 2025-09-29


In [12]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import re

HEADERS = BROWSER_HEADERS

# Fetch DataMapper page to find Fiscal Rules Excel URL
fr_page = requests.get(
    'https://www.imf.org/external/datamapper/fiscalrules/map/map.htm',
    headers=HEADERS,
    timeout=30
)
print(f"Status: {fr_page.status_code}")

# Search for Excel/CSV/download links
xlsx_links = re.findall(r'https?://[^\s"\'<>]+\.xlsx', fr_page.text)
ashx_links = re.findall(r'https?://[^\s"\'<>]+\.ashx', fr_page.text)
media_links = re.findall(r'/-/media/[^\s"\'<>]+', fr_page.text)

print(f"XLSX links: {xlsx_links[:5]}")
print(f"ASHX links: {ashx_links[:5]}")
print(f"Media links (fiscal/rules): {[l for l in media_links if 'fiscal' in l.lower() or 'rules' in l.lower()][:5]}")
print(f"\nFirst 1000 chars:")
print(fr_page.text[:1000])

Status: 403
XLSX links: []
ASHX links: []
Media links (fiscal/rules): []

First 1000 chars:
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;imf&#46;org&#47;external&#47;datamapper&#47;fiscalrules&#47;map&#47;map&#46;htm" on this server.<P>
Reference&#32;&#35;18&#46;c3aa3717&#46;1781288360&#46;db11f77
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;c3aa3717&#46;1781288360&#46;db11f77</P>
</BODY>
</HTML>

